In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import userdata
userdata.get('HF_TOKEN')

'HF_TOKEN'

In [ ]:
!pip install -q pythainlp deepcut
!pip install -U marisa-trie

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 29.9 MB/s eta 0:00:00


In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
from tqdm.auto import tqdm

from pythainlp.tokenize import word_tokenize

from datasets import load_dataset
from marisa_trie import Trie
import deepcut

In [ ]:
!unzip -q '/content/opend_lst20_corpus.zip'

In [ ]:
import glob
from datasets import Dataset, DatasetDict

def load_lst20_local(path_pattern):
    all_tokens = []
    for file_path in glob.glob(path_pattern):
        with open(file_path, 'r', encoding='utf-8') as f:
            current_sentence = []
            for line in f:
                line = line.strip()
                if not line:
                    if current_sentence:
                        all_tokens.append(current_sentence)
                        current_sentence = []
                    continue
                parts = line.split('\t')
                if len(parts) > 0:
                    current_sentence.append(parts[0])
            if current_sentence:
                all_tokens.append(current_sentence)
    return Dataset.from_dict({'tokens': all_tokens})

lst20_dataset = DatasetDict({
    'train': load_lst20_local('/content/LST20_Corpus/train/*.txt'),
    'test': load_lst20_local('/content/LST20_Corpus/test/*.txt'),
    'validation': load_lst20_local('/content/LST20_Corpus/eval/*.txt')
})

In [ ]:
lst20_dataset

DatasetDict({
    train: Dataset({
        features: ['tokens'],
        num_rows: 63310
    })
    test: Dataset({
        features: ['tokens'],
        num_rows: 5250
    })
    validation: Dataset({
        features: ['tokens'],
        num_rows: 5620
    })
})

In [ ]:
word_set = set()
splits = ['train', 'validation', 'test']
for split in splits:
    for words in lst20_dataset[split]['tokens']:
        for word in words:
            word_set.add(word)

print(f"All word: {len(word_set)}")
lst20_trie = Trie(word_set)

All word: 54770


In [ ]:
with open("/content/ws_test.txt","r") as f:
    test_doc = f.read()

with open("/content/ws_list.txt") as f:
    ws_list = eval(f.read())

In [ ]:
print(len(test_doc))
print(f"{test_doc[:200]} ...")

37248
ที่ยังสถานการณ์ยังไม่คลี่คลายอาจส่งผลกระทบการค้าชายแดนไทย - กัมพูชา ว่า เท่าที่เข้าไปดำเนินการตรวจสอบยังไม่พบว่ามีการปิดด่านบริเวณดังกล่าว และที่สำคัญการค้าระหว่างไทยและกัมพูชาส่วนใหญ่จะมีปริมาณมากที่ ...


In [ ]:
ws_list

['B_WORD', 'E_WORD', 'I_WORD']

In [ ]:
sample_sub = pd.read_csv("/content/ws_sample_submission.csv")
sample_sub

,Id,Predicted
0,1,B_WORD
1,2,I_WORD
2,3,E_WORD
3,4,NaN
4,5,NaN
...,...,...
35177,37244,NaN
35178,37245,NaN
35179,37246,NaN
35180,37247,NaN


In [ ]:
word_tokenize(test_doc[:200])

['ที่',
 'ยัง',
 'สถานการณ์',
 'ยัง',
 'ไม่',
 'คลี่คลาย',
 'อาจ',
 'ส่ง',
 'ผลกระทบ',
 'การค้า',
 'ชายแดน',
 'ไทย',
 ' ',
 '-',
 ' ',
 'กัมพูชา',
 ' ',
 'ว่า',
 ' ',
 'เท่าที่',
 'เข้าไป',
 'ดำเนินการ',
 'ตรวจสอบ',
 'ยัง',
 'ไม่',
 'พบ',
 'ว่า',
 'มี',
 'การ',
 'ปิด',
 'ด่าน',
 'บริเวณ',
 'ดังกล่าว',
 ' ',
 'และ',
 'ที่',
 'สำคัญ',
 'การค้า',
 'ระหว่าง',
 'ไทย',
 'และ',
 'กัมพูชา',
 'ส่วนใหญ่',
 'จะ',
 'มี',
 'ปริมาณมาก',
 'ที่']

In [ ]:
len(test_doc.replace(' ', ''))

35182

In [ ]:
test_doc_clean = test_doc.replace(' ', '')

In [ ]:
more_vocab = [
            "ทรลักษ์", "อนุพงษ์เผ่าจินดา", "ลองวิสาโล", "เนียงพาด", "พอลซาเรือน", "จีบีซี", "วลิตโรจน", "วลิต", "วิตก", "วีรชัย", "พลาศรัย", "เอี่ยมฐานนท์",
              "โมหำหมัด", "ซอรีสะมะแอ", "สุวรรณเกษการ", "ซอรี", "กวินตรา", "โพธิจักร", "ภู่ขวัญเมือง", "ไฟติ้ง", "เจริญวัชรวิทย์", "ปลั่งพินิจ", "ลุกทุ่ง", "มหาชน",
              "เมโท", "หยิง", "ตาราบารอฟ", "เฮย", "อุสซูริส", "อามูร์มา", "เหยหลง", "วูซูลิเจียง", "วานิจ", "ปิณฑวนิช", "โกสัย", 'ดาตอร์ปิโด', 'ดารณี', 'ศิลปกุล',
              'ยกุล', 'สืบวงษ์ลี', 'กระจอก', 'เจียโก', 'วิรายุดา', 'วงษ์เทศ', 'ขอม', 'ธม', 'เสภา', 'เสภา', 'ออเคสตรา' ,'อรรถจักร', 'สัตยานุรักษ์'
             ]
lst20_words = list(word_set)
lst20_words += more_vocab
lst20_trie = Trie(set(lst20_words))

In [ ]:
from pythainlp.util import Trie as PyThaiNlpTrie
from tqdm.auto import tqdm

# Create a PyThaiNLP compatible Trie
custom_dict = PyThaiNlpTrie(set(lst20_words))

id_count = 0
words = word_tokenize(test_doc, engine='deepcut', keep_whitespace=False)
all_space_idx = [idx for idx, c in enumerate(test_doc) if c==" "]
encode_pairs = [] #[id, word_segment]
oov_words = []

# clear space
words_clear = []
for word in words:
    words_clear += word.split(" ")

def has_numbers(inputString):
    return any(char.isdigit() for char in inputString)

for i, word in tqdm(enumerate(words_clear), total=len(words_clear)):

    if word not in lst20_words:
        oov_words.append(word)
        # Use the PyThaiNLP Trie object
        new_words = word_tokenize(word, engine='newmm', custom_dict=custom_dict)
        if has_numbers(word):
            new_words = word_tokenize(word, engine='longest', custom_dict=custom_dict)

        for n_word in new_words:
            for idx, c in enumerate(n_word):
                id_count += 1
                if id_count in all_space_idx:
                    id_count += 1

                if idx == 0:
                    encode_pairs.append([id_count, 'B_WORD'])
                elif (idx == len(n_word)-1) and (len(n_word) > 1):
                    encode_pairs.append([id_count, 'E_WORD'])
                else:
                    encode_pairs.append([id_count, 'I_WORD'])

    else:
        for idx, c in enumerate(word):
            id_count += 1
            if id_count in all_space_idx:
                id_count += 1

            if idx == 0:
                encode_pairs.append([id_count, 'B_WORD'])
            elif (idx == len(word)-1) and (len(word) > 1):
                encode_pairs.append([id_count, 'E_WORD'])
            else:
                encode_pairs.append([id_count, 'I_WORD'])

1164/1164 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step


  0%|          | 0/8074 [00:00<?, ?it/s]

In [ ]:
pred = pd.DataFrame(encode_pairs)
pred.columns = ['Id', 'Predicted']

In [ ]:
pred

,Id,Predicted
0,1,B_WORD
1,2,I_WORD
2,3,E_WORD
3,4,B_WORD
4,5,I_WORD
...,...,...
35177,37244,E_WORD
35178,37245,B_WORD
35179,37246,I_WORD
35180,37247,I_WORD


In [ ]:
len(words_clear)

8074

In [ ]:
sample_sub['Predicted'] = pred['Predicted']

In [ ]:
sample_sub

,Id,Predicted
0,1,B_WORD
1,2,I_WORD
2,3,E_WORD
3,4,B_WORD
4,5,I_WORD
...,...,...
35177,37244,E_WORD
35178,37245,B_WORD
35179,37246,I_WORD
35180,37247,I_WORD


In [ ]:
sample_sub.to_csv("submiss.csv", index=False)